# EA2 - Actividad 2.3: Spark SQL

## Objetivos
- Ejecutar consultas SQL nativas sobre DataFrames de Spark
- Crear y usar vistas temporales
- Dominar joins, subqueries y CTEs en Spark SQL
- Aplicar window functions (ROW_NUMBER, RANK, LAG, LEAD, SUM OVER)
- Comparar SQL vs DataFrame API

## Conceptos Clave

### Spark SQL Engine

Spark SQL permite ejecutar consultas SQL estandar sobre DataFrames. Internamente, Spark usa el **Catalyst Optimizer** para optimizar las consultas, independientemente de si se escriben en SQL o con la DataFrame API.

**Flujo de trabajo:**
1. Registrar un DataFrame como **vista temporal** con `createOrReplaceTempView("nombre")`
2. Ejecutar consultas con `spark.sql("SELECT ...")`
3. El resultado es un nuevo DataFrame

### SQL vs DataFrame API

| SQL | DataFrame API |
|-----|---------------|
| `SELECT col FROM tabla` | `df.select("col")` |
| `WHERE col > 10` | `df.filter(F.col("col") > 10)` |
| `GROUP BY col` | `df.groupBy("col")` |
| `ORDER BY col DESC` | `df.orderBy(F.col("col").desc())` |
| `JOIN t2 ON t1.id = t2.id` | `df1.join(df2, "id")` |

Ambas producen el **mismo plan de ejecucion** gracias al Catalyst Optimizer.

### Window Functions

Las funciones de ventana permiten realizar calculos sobre un conjunto de filas relacionadas con la fila actual, sin colapsar los datos (a diferencia de GROUP BY).

| Funcion | Descripcion |
|---------|-------------|
| `ROW_NUMBER()` | Numero secuencial unico por particion |
| `RANK()` | Ranking con empates (salta posiciones) |
| `DENSE_RANK()` | Ranking con empates (sin saltar posiciones) |
| `LAG(col, n)` | Valor de `n` filas anteriores |
| `LEAD(col, n)` | Valor de `n` filas siguientes |
| `SUM() OVER` | Suma acumulada sobre la ventana |

## Setup

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import *

spark = SparkSession.builder \
    .appName("EA2_03_spark_sql") \
    .master("local[*]") \
    .getOrCreate()

print(f"Spark version: {spark.version}")

## 1. Cargar Datos y Crear Vistas Temporales

Para usar SQL en Spark, primero debemos registrar los DataFrames como **vistas temporales**. Esto las hace accesibles desde `spark.sql()`.

In [ ]:
# Leer los datasets
df_sales = spark.read.csv("/home/jovyan/datos/sales.csv", header=True, inferSchema=True)
df_stores = spark.read.csv("/home/jovyan/datos/stores.csv", header=True, inferSchema=True)

# Crear vistas temporales (como tablas en SQL)
df_sales.createOrReplaceTempView("ventas")
df_stores.createOrReplaceTempView("tiendas")

print("Vistas temporales creadas: 'ventas' y 'tiendas'")
print(f"\nventas: {df_sales.count()} filas")
print(f"tiendas: {df_stores.count()} filas")

In [ ]:
# Verificar que las vistas funcionan con una query simple
spark.sql("SELECT * FROM ventas LIMIT 5").show()
spark.sql("SELECT * FROM tiendas").show()

## 2. Queries Basicos: SELECT, WHERE, GROUP BY, HAVING, ORDER BY

Las consultas SQL en Spark siguen la sintaxis SQL estandar.

In [ ]:
# SELECT + WHERE: filtrar ventas mayores a 100,000
spark.sql("""
    SELECT Store, Dept, Date, Weekly_Sales
    FROM ventas
    WHERE Weekly_Sales > 100000
    ORDER BY Weekly_Sales DESC
""").show(10)

In [ ]:
# GROUP BY + HAVING: tiendas con ventas totales mayores a 1,000,000
spark.sql("""
    SELECT Store, SUM(Weekly_Sales) as total
    FROM ventas
    GROUP BY Store
    HAVING SUM(Weekly_Sales) > 1000000
    ORDER BY total DESC
""").show()

In [ ]:
# Funciones de agregacion
spark.sql("""
    SELECT 
        Store,
        COUNT(*) as num_registros,
        ROUND(AVG(Weekly_Sales), 2) as promedio_ventas,
        ROUND(MIN(Weekly_Sales), 2) as min_ventas,
        ROUND(MAX(Weekly_Sales), 2) as max_ventas,
        ROUND(SUM(Weekly_Sales), 2) as total_ventas
    FROM ventas
    GROUP BY Store
    ORDER BY total_ventas DESC
    LIMIT 10
""").show()

## 3. Joins en SQL

Los joins combinan datos de dos tablas basandose en una condicion. Spark SQL soporta todos los tipos de join estandar.

In [ ]:
# INNER JOIN: combinar ventas con informacion de tiendas
spark.sql("""
    SELECT v.Store, t.Type, t.Size, SUM(v.Weekly_Sales) as total
    FROM ventas v
    INNER JOIN tiendas t ON v.Store = t.Store
    GROUP BY v.Store, t.Type, t.Size
    ORDER BY total DESC
""").show(10)

In [ ]:
# LEFT JOIN: incluir tiendas aunque no tengan ventas
spark.sql("""
    SELECT t.Store, t.Type, t.Size, 
           COALESCE(SUM(v.Weekly_Sales), 0) as total_ventas
    FROM tiendas t
    LEFT JOIN ventas v ON t.Store = v.Store
    GROUP BY t.Store, t.Type, t.Size
    ORDER BY t.Store
""").show()

## 4. Subqueries: IN y EXISTS

Las subqueries permiten usar el resultado de una consulta dentro de otra.

In [ ]:
# Subquery con IN: ventas de tiendas tipo A
spark.sql("""
    SELECT Store, Dept, Weekly_Sales
    FROM ventas
    WHERE Store IN (
        SELECT Store FROM tiendas WHERE Type = 'A'
    )
    ORDER BY Weekly_Sales DESC
    LIMIT 10
""").show()

In [ ]:
# Subquery con EXISTS: tiendas que tienen al menos una venta > 100,000
spark.sql("""
    SELECT t.Store, t.Type, t.Size
    FROM tiendas t
    WHERE EXISTS (
        SELECT 1 FROM ventas v 
        WHERE v.Store = t.Store AND v.Weekly_Sales > 100000
    )
""").show()

## 5. CTEs (Common Table Expressions)

Las CTEs (`WITH`) permiten definir subconsultas nombradas que se pueden reusar dentro de la query principal. Son mas legibles que subqueries anidadas.

In [ ]:
# CTE: encontrar tiendas con ventas totales > 50,000,000
spark.sql("""
    WITH ventas_por_tienda AS (
        SELECT Store, SUM(Weekly_Sales) as total
        FROM ventas
        GROUP BY Store
    )
    SELECT * FROM ventas_por_tienda 
    WHERE total > 50000000
    ORDER BY total DESC
""").show()

In [ ]:
# CTE con multiples subconsultas
spark.sql("""
    WITH ventas_tienda AS (
        SELECT Store, SUM(Weekly_Sales) as total_ventas
        FROM ventas
        GROUP BY Store
    ),
    info_tienda AS (
        SELECT Store, Type, Size
        FROM tiendas
    )
    SELECT i.Store, i.Type, i.Size, v.total_ventas
    FROM ventas_tienda v
    JOIN info_tienda i ON v.Store = i.Store
    ORDER BY v.total_ventas DESC
    LIMIT 10
""").show()

## 6. Window Functions: ROW_NUMBER y RANK

Las funciones de ventana operan sobre un conjunto de filas (la "ventana") definido por `PARTITION BY` y `ORDER BY`, sin colapsar los datos.

**Sintaxis:** `FUNCION() OVER (PARTITION BY col ORDER BY col)`

In [ ]:
# ROW_NUMBER y RANK: rankear departamentos por ventas dentro de cada tienda
spark.sql("""
    SELECT Store, Dept, Weekly_Sales,
        ROW_NUMBER() OVER (PARTITION BY Store ORDER BY Weekly_Sales DESC) as row_num,
        RANK() OVER (PARTITION BY Store ORDER BY Weekly_Sales DESC) as rank_con_empates
    FROM ventas
""").show(20)

In [ ]:
# Ejemplo practico: obtener el departamento con mas ventas por tienda
spark.sql("""
    WITH ranking AS (
        SELECT Store, Dept, SUM(Weekly_Sales) as total,
            ROW_NUMBER() OVER (PARTITION BY Store ORDER BY SUM(Weekly_Sales) DESC) as rn
        FROM ventas
        GROUP BY Store, Dept
    )
    SELECT Store, Dept, total
    FROM ranking
    WHERE rn = 1
    ORDER BY total DESC
    LIMIT 10
""").show()

## 7. Mas Window Functions: LAG, LEAD, SUM OVER

- **LAG(col, n):** Accede al valor de `n` filas **anteriores**
- **LEAD(col, n):** Accede al valor de `n` filas **siguientes**
- **SUM() OVER:** Suma acumulada sobre la ventana

In [ ]:
# LAG y LEAD: comparar ventas con la semana anterior/siguiente
# Ejemplo con tienda 1, departamento 1
spark.sql("""
    SELECT Store, Date, Weekly_Sales,
        LAG(Weekly_Sales, 1) OVER (PARTITION BY Store ORDER BY Date) as venta_anterior,
        LEAD(Weekly_Sales, 1) OVER (PARTITION BY Store ORDER BY Date) as venta_siguiente,
        Weekly_Sales - LAG(Weekly_Sales, 1) OVER (PARTITION BY Store ORDER BY Date) as diferencia
    FROM ventas
    WHERE Store = 1 AND Dept = 1
    ORDER BY Date
""").show(15)

In [ ]:
# SUM OVER: suma acumulada de ventas
spark.sql("""
    SELECT Store, Date, Weekly_Sales,
        SUM(Weekly_Sales) OVER (
            PARTITION BY Store 
            ORDER BY Date 
            ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
        ) as venta_acumulada
    FROM ventas
    WHERE Store = 1 AND Dept = 1
    ORDER BY Date
""").show(15)

---
## Ejercicios

Ahora es tu turno de practicar. Completa los siguientes ejercicios.

In [ ]:
# =============================================================
# EJERCICIO 1: Top 5 tiendas por ventas totales
# =============================================================
# TODO: Escribe una query SQL que muestre:
#   - Store (numero de tienda)
#   - Type (tipo de tienda, del join con tiendas)
#   - total_ventas (suma de Weekly_Sales)
# Ordenado de mayor a menor, mostrando solo las top 5
#
# Pista: Necesitas un JOIN entre ventas y tiendas,
#        GROUP BY y ORDER BY, con LIMIT 5

# Escribe tu codigo aqui:
# spark.sql("""
#     SELECT ...
#     FROM ventas v
#     ...JOIN tiendas t ON ...
#     GROUP BY ...
#     ORDER BY ...
#     LIMIT 5
# """).show()


In [ ]:
# =============================================================
# EJERCICIO 2: Rankear departamentos por ventas dentro de cada tienda
# =============================================================
# TODO: Usa RANK() con PARTITION BY Store para rankear
#   los departamentos dentro de cada tienda por ventas totales.
#   Muestra: Store, Dept, total_ventas, ranking
#   Filtra solo los top 3 por tienda (ranking <= 3)
#
# Pista: Usa un CTE para calcular el ranking y luego filtra
#   WITH ranking AS (
#       SELECT ..., RANK() OVER (PARTITION BY Store ORDER BY ... DESC) as rnk
#       FROM ventas
#       GROUP BY Store, Dept
#   )
#   SELECT * FROM ranking WHERE rnk <= 3

# Escribe tu codigo aqui:


In [ ]:
# =============================================================
# EJERCICIO 3: Diferencia de ventas con semana anterior (Tienda 1)
# =============================================================
# TODO: Usa LAG() para calcular la diferencia de ventas
#   respecto a la semana anterior para la tienda 1 (Store = 1).
#   Muestra: Store, Dept, Date, Weekly_Sales, venta_anterior, diferencia
#   Filtra solo para Dept = 1 para simplificar
#   Ordena por Date
#
# Pista:
#   LAG(Weekly_Sales, 1) OVER (PARTITION BY Store ORDER BY Date)
#   diferencia = Weekly_Sales - LAG(...)

# Escribe tu codigo aqui:


---
## Desafio Extra (Opcional)

**Para estudiantes avanzados:**

Query complejo combinando CTEs, window functions y joins.

In [ ]:
# =============================================================
# DESAFIO: Query avanzado con CTEs + Window + Joins
# =============================================================
# TODO: Para cada tipo de tienda (A, B, C), mostrar:
#   1. La tienda con mayores ventas totales
#   2. Su ranking dentro de su tipo
#   3. Cuanto supera al promedio de ventas de su tipo
#
# Pasos sugeridos:
#   CTE 1: Calcular ventas totales por tienda (join con tiendas para obtener Type)
#   CTE 2: Calcular promedio por tipo de tienda
#   CTE 3: Rankear tiendas dentro de cada tipo con RANK()
#   Query final: Filtrar ranking = 1 y calcular diferencia con promedio
#
# Columnas esperadas: Type, Store, total_ventas, promedio_tipo, 
#                     diferencia_sobre_promedio, ranking
#
# Pista: Puedes encadenar multiples CTEs con comas:
#   WITH cte1 AS (...), cte2 AS (...), cte3 AS (...)
#   SELECT ...

# Escribe tu codigo aqui:


---
## Resumen

En esta actividad aprendimos:

1. **Vistas temporales:** `df.createOrReplaceTempView("nombre")` para habilitar SQL
2. **Queries basicos:** SELECT, WHERE, GROUP BY, HAVING, ORDER BY en Spark SQL
3. **Joins:** INNER JOIN y LEFT JOIN para combinar tablas
4. **Subqueries:** IN y EXISTS para consultas anidadas
5. **CTEs:** `WITH nombre AS (...)` para subconsultas legibles y reutilizables
6. **Window functions:** ROW_NUMBER, RANK para rankings sin colapsar datos
7. **LAG/LEAD:** Acceder a filas anteriores/siguientes para calcular diferencias
8. **SUM OVER:** Sumas acumuladas sobre ventanas ordenadas

In [ ]:
# Detener la SparkSession al finalizar
spark.stop()
print("SparkSession detenida correctamente.")